# Калибровка нормативов операций

Этот notebook анализирует исторические данные из `event_logs` / `mart_daily_losses`
и предлагает скорректированные нормативы (`norm_seconds`) на основе статистики.

Запуск:
- Вручную через Jupyter Lab
- Автоматически через Airflow DAG `ml_retrain_pipeline` (papermill)

## Параметры (papermill)

In [ ]:
# Параметры (перезаписываются papermill)
CLICKHOUSE_HOST = "localhost"
CLICKHOUSE_PORT = 8123
CLICKHOUSE_DB = "processfix"
CLICKHOUSE_USER = "default"
CLICKHOUSE_PASSWORD = ""
MINIO_ENDPOINT = "localhost:9000"
MINIO_ACCESS_KEY = "minioadmin"
MINIO_SECRET_KEY = "minioadmin"
LOOKBACK_DAYS = 30
OUTPUT_BUCKET = "processfix-artifacts"
OUTPUT_KEY = "norm_calibration/latest.csv"

In [ ]:
import clickhouse_connect
import pandas as pd
import numpy as np
from datetime import datetime, timedelta

client = clickhouse_connect.get_client(
    host=CLICKHOUSE_HOST,
    port=CLICKHOUSE_PORT,
    database=CLICKHOUSE_DB,
    username=CLICKHOUSE_USER,
    password=CLICKHOUSE_PASSWORD,
)
print(f"Connected to ClickHouse: {CLICKHOUSE_HOST}:{CLICKHOUSE_PORT}/{CLICKHOUSE_DB}")

## 1. Загрузка данных

In [ ]:
since = datetime.utcnow() - timedelta(days=LOOKBACK_DAYS)

query = """
SELECT
    operation_name,
    duration_seconds,
    toDate(start_time) AS event_date
FROM event_logs
WHERE start_time >= %(since)s
  AND duration_seconds > 0
  AND duration_seconds < 86400
"""
result = client.query(query, parameters={"since": since})
df = pd.DataFrame(result.result_rows, columns=["operation_name", "duration_seconds", "event_date"])
print(f"Загружено {len(df):,} записей за последние {LOOKBACK_DAYS} дней")
df.head()

## 2. Текущие нормативы

In [ ]:
norms_result = client.query("SELECT operation_name, norm_seconds FROM dim_process_norms FINAL")
norms_df = pd.DataFrame(norms_result.result_rows, columns=["operation_name", "current_norm_sec"])
norms_df

## 3. Статистический анализ

In [ ]:
stats = df.groupby("operation_name")["duration_seconds"].agg(
    ["count", "mean", "median", "std", lambda x: np.percentile(x, 75)]
).rename(columns={"<lambda_0>": "p75"})
stats.columns = ["event_count", "mean_sec", "median_sec", "std_sec", "p75_sec"]
stats = stats.round(1).reset_index()

# Рекомендуемый норматив: медиана + 0.5*std (покрывает ~70% случаев)
stats["recommended_norm_sec"] = (stats["median_sec"] + 0.5 * stats["std_sec"]).round(0).astype(int)

comparison = stats.merge(norms_df, on="operation_name", how="left")
comparison["delta_sec"] = comparison["recommended_norm_sec"] - comparison["current_norm_sec"]
comparison["delta_pct"] = ((comparison["delta_sec"] / comparison["current_norm_sec"]) * 100).round(1)

print("Сравнение текущих и рекомендуемых нормативов:")
comparison

## 4. Выгрузка рекомендаций в MinIO

In [ ]:
import io
from minio import Minio

csv_buffer = io.BytesIO()
comparison.to_csv(csv_buffer, index=False, encoding="utf-8")
csv_buffer.seek(0)

minio_client = Minio(
    MINIO_ENDPOINT,
    access_key=MINIO_ACCESS_KEY,
    secret_key=MINIO_SECRET_KEY,
    secure=False,
)

minio_client.put_object(
    OUTPUT_BUCKET,
    OUTPUT_KEY,
    csv_buffer,
    length=csv_buffer.getbuffer().nbytes,
    content_type="text/csv",
)
print(f"Рекомендации сохранены: s3://{OUTPUT_BUCKET}/{OUTPUT_KEY}")

## 5. Итог

Рекомендации сохранены в MinIO. Для применения:
1. Проверить `comparison` DataFrame выше.
2. При согласовании — обновить `process_norms` через API или напрямую в PG.
3. Перезапустить `sync_dims_to_clickhouse` для актуализации CH.